In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile
import os

# Assuming the uploaded file is named 'archive.zip'
zip_file_path = '/content/drive/MyDrive/dataset/archive.zip'

# Check if the zip file exists
if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall('.') # Extract all contents to the current directory
    print(f"Successfully unzipped '{zip_file_path}'")
    print("Extracted files:")
    for filename in zip_ref.namelist():
        print(f"- {filename}")
else:
    print(f"Error: Zip file '{zip_file_path}' not found.")

Successfully unzipped '/content/drive/MyDrive/dataset/archive.zip'
Extracted files:
- faker_employee.csv


In [5]:
!pip install pyspark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SQL_Practice") \
    .master("local[*]") \
    .getOrCreate()

In [6]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/faker_employee.csv")

df.createOrReplaceTempView("employees")

In [7]:
spark.sql("SELECT * FROM employees LIMIT 5").show()

+------------------+----------+---------+----------+------+------------+--------------------+--------------------+
|       Employee ID|First Name|Last Name|Department|Salary|Joining Date|            Email ID|             Address|
+------------------+----------+---------+----------+------+------------+--------------------+--------------------+
|                 1|  Nicholas|   Martin|     Sales| 50819|  2022-11-16|dannymays@example...|14632 Ashley Traf...|
|  Lake Katrinaside| MH 96450"|     NULL|      NULL|  NULL|        NULL|                NULL|                NULL|
|                 2|       Joy|   Miller|     Legal| 58024|  2023-02-03|heather59@example...|00670 Oliver Harbors|
|Port Andrewchester| TN 14223"|     NULL|      NULL|  NULL|        NULL|                NULL|                NULL|
|                 3|    Thomas|     Roth|       R&D| 54572|  2021-08-18|garyhogan@example...|  3024 John Turnpike|
+------------------+----------+---------+----------+------+------------+--------

In [8]:
spark.sql("""
    SELECT
        SUM(CASE WHEN Department IS NULL THEN 1 ELSE 0 END) as null_department,
        SUM(CASE WHEN Salary IS NULL THEN 1 ELSE 0 END) as null_salary,
        SUM(CASE WHEN `Email ID` IS NULL THEN 1 ELSE 0 END) as null_email
    FROM employees
""").show()

+---------------+-----------+----------+
|null_department|null_salary|null_email|
+---------------+-----------+----------+
|        1000000|    1000000|   1000000|
+---------------+-----------+----------+



In [9]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW employees_clean AS
    SELECT * FROM employees
    WHERE Salary IS NOT NULL AND Department IS NOT NULL
""")

spark.sql("SELECT COUNT(*) as clean_row_count FROM employees_clean").show()

+---------------+
|clean_row_count|
+---------------+
|        1000000|
+---------------+



In [10]:
spark.sql("""
    SELECT * FROM employees_clean
    WHERE Salary > 70000
""").show(5)

+-----------+----------+---------+----------------+------+------------+--------------------+--------------------+
|Employee ID|First Name|Last Name|      Department|Salary|Joining Date|            Email ID|             Address|
+-----------+----------+---------+----------------+------+------------+--------------------+--------------------+
|          4|  Brittney|   Morgan|           Sales| 72174|  2022-10-03|wcervantes@exampl...|374 Cook Dam Suit...|
|         15|    Andrew|    Joyce| Human Resources| 74864|  2023-03-18| katie91@example.org|5957 Samantha Row...|
|         16| Stephanie|    Smith|Customer Service| 72876|  2023-05-22|  john34@example.net|  PSC 4767, Box 4987|
|         19|     Karen| Mitchell|      Operations| 75018|  2023-09-27|ramirezkristen@ex...|    955 Hector Manor|
|         24|   Michael| Figueroa|           Legal| 76447|  2022-12-07|  sriley@example.net|    6164 Troy Estate|
+-----------+----------+---------+----------------+------+------------+-----------------

In [11]:
spark.sql("""
    SELECT Department, COUNT(*) as emp_count, AVG(Salary) as avg_salary
    FROM employees_clean
    GROUP BY Department
    ORDER BY avg_salary DESC
""").show()

+--------------------+---------+------------------+
|          Department|emp_count|        avg_salary|
+--------------------+---------+------------------+
|Training and Deve...|    49859| 65050.49349164644|
|    Customer Service|    49813|   65046.247204545|
|      Administration|    50195| 65028.46020519972|
|           Logistics|    50078| 65015.32018051839|
|                  QA|    50203| 65010.47891560265|
|               Sales|    50103| 65007.39127397561|
|                  IT|    50048| 65006.39695891944|
|          Operations|    49781| 65005.79460034953|
|             Finance|    50446| 65001.49437021766|
|  Strategic Planning|    50268|64997.413006286304|
|               Legal|    50179| 64992.07754239821|
|Facilities Manage...|    49774|64986.630268815046|
|           Marketing|    50020| 64985.23884446221|
|          Compliance|    50089|64979.198247120126|
|Corporate Communi...|    49698|64978.049559338404|
|                  PR|    50004| 64975.77049836013|
|     Human 

In [12]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW departments AS
    SELECT * FROM VALUES
        ('Sales', 500000),
        ('Legal', 300000),
        ('R&D', 800000),
        ('Training and Development', 200000)
    AS departments(Department, Budget)
""")

spark.sql("""
    SELECT e.`First Name`, e.Department, e.Salary, d.Budget
    FROM employees_clean e
    JOIN departments d ON e.Department = d.Department
""").show(5)

+----------+--------------------+------+------+
|First Name|          Department|Salary|Budget|
+----------+--------------------+------+------+
|  Nicholas|               Sales| 50819|500000|
|       Joy|               Legal| 58024|300000|
|    Thomas|                 R&D| 54572|800000|
|  Brittney|               Sales| 72174|500000|
| Katherine|Training and Deve...| 52848|200000|
+----------+--------------------+------+------+
only showing top 5 rows


In [13]:
spark.sql("""
    SELECT `First Name`, Department, Salary,
           RANK() OVER (PARTITION BY Department ORDER BY Salary DESC) as salary_rank
    FROM employees_clean
""").show(10)

+----------+--------------------+------+-----------+
|First Name|          Department|Salary|salary_rank|
+----------+--------------------+------+-----------+
|     Jacob|Business Development| 80000|          1|
|    Shelby|Business Development| 80000|          1|
|     Jason|Business Development| 79999|          3|
|    Angela|Business Development| 79999|          3|
|      Jean|Business Development| 79999|          3|
|   Matthew|Business Development| 79999|          3|
|   Gregory|Business Development| 79999|          3|
|   Gabriel|Business Development| 79998|          8|
|  Lawrence|Business Development| 79997|          9|
|      Seth|Business Development| 79996|         10|
+----------+--------------------+------+-----------+
only showing top 10 rows


In [14]:
spark.sql("""
    SELECT `First Name`, Department, Salary,
           SUM(Salary) OVER (
               PARTITION BY Department
               ORDER BY Salary DESC
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) as running_total
    FROM employees_clean
""").show(10)

+----------+--------------------+------+-------------+
|First Name|          Department|Salary|running_total|
+----------+--------------------+------+-------------+
|     Jacob|Business Development| 80000|        80000|
|    Shelby|Business Development| 80000|       160000|
|     Jason|Business Development| 79999|       239999|
|    Angela|Business Development| 79999|       319998|
|      Jean|Business Development| 79999|       399997|
|   Matthew|Business Development| 79999|       479996|
|   Gregory|Business Development| 79999|       559995|
|   Gabriel|Business Development| 79998|       639993|
|  Lawrence|Business Development| 79997|       719990|
|      Seth|Business Development| 79996|       799986|
+----------+--------------------+------+-------------+
only showing top 10 rows
